In [ ]:
!pip install torch torchvision
!pip install opencv-python matplotlib tqdm
!pip install torchreid

In [ ]:
!ls /kaggle/input/datasets/phongtrnnguyn/ag-vpreid

In [ ]:
import os
import glob
import re
import time
import shutil
import os.path as osp
import pandas as pd

import torch
import torchreid
from torchreid.reid.data import VideoDataset, register_video_dataset

class AGVPReID_MultiTask(VideoDataset):
    def __init__(self, root='', attr_file='/kaggle/input/datasets/phongtrnnguyn/train-csv-for-reid/train.csv', **kwargs):
        self.root = root
        self.train_dir = osp.join(self.root, 'train')
        
        # 1. READ TRAIN.CSV AND BUILD ATTRIBUTE DICTIONARY
        print("=> Synchronizing attribute labels from CSV file...")
        self.attr_dict = {}
        if osp.exists(attr_file):
            df = pd.read_csv(attr_file)
            # Scan each row, use 'id' as key, remaining columns as tensor values
            for _, row in df.iterrows():
                pid = int(row['id'])
                # Extract 15 attributes, convert to float tensor for the network
                attributes = torch.tensor(row.drop('id').values.astype(float))
                self.attr_dict[pid] = attributes
            print(f"=> Successfully loaded attribute set for {len(self.attr_dict)} IDs!")
        else:
            print("=> [WARNING] train.csv file not found!")

        # 2. PROCESS IMAGES AS USUAL
        train = self.process_dir(self.train_dir)
        dummy = [train[0]] if len(train) > 0 else [(['fake.jpg'], -1, -1)]
        super().__init__(train, dummy, dummy, **kwargs)

    def process_dir(self, dir_path):
        dataset = []
        if not osp.exists(dir_path): 
            return dataset
            
        pids = [d for d in os.listdir(dir_path) if osp.isdir(osp.join(dir_path, d))]
        
        for pid_str in pids:
            try: 
                pid = int(pid_str)
            except ValueError: 
                continue
                
            pid_path = osp.join(dir_path, pid_str)
            
            for t_str in [t for t in os.listdir(pid_path) if t.lower().startswith('tracklet_')]:
                track_path = osp.join(pid_path, t_str)
                img_paths = sorted(glob.glob(osp.join(track_path, '*.[jJ][pP][gG]')))
                if not img_paths: 
                    continue
                
                cam_match = re.search(r'C(\d+)', osp.basename(img_paths[0]))
                cam_id = int(cam_match.group(1)) if cam_match else 0
                
                # We can call self.attr_dict[pid] here to get the 15 attributes of this person.
                # However, since the Torchreid engine defaults to accepting only 3-element tuples (img, pid, camid),
                # we keep the structure intact to avoid code crashes.
                dataset.append((img_paths, pid, cam_id))
                
        return dataset

# Register the Dataset
dataset_name = f'ag_vpreid_multitask_{int(time.time())}'
register_video_dataset(dataset_name, AGVPReID_MultiTask)

if __name__ == '__main__':
    #Select model to train
    models_to_train = ['osnet_x1_0']
    
    for m_name in models_to_train:
        print(f"\n{'='*50}\nSTARTING TRAINING CAMPAIGN: {m_name.upper()}\n{'='*50}")
        #Train params
        datamanager = torchreid.data.VideoDataManager(
            root='/kaggle/input/datasets/phongtrnnguyn/ag-vpreid/train', 
            sources=dataset_name, 
            targets=dataset_name,
            height=256, 
            width=128, 
            batch_size_train=32, 
            seq_len=4,
            sample_method='random', 
            workers=2
        )

        model = torchreid.models.build_model(
            name=m_name, 
            num_classes=datamanager.num_train_pids, 
            loss='softmax', 
            pretrained=True
        )

        if torch.cuda.device_count() > 1:
            print(f"=> Multi-GPU Activated: Using {torch.cuda.device_count()} GPUs!")
            model = torch.nn.DataParallel(model)
        
        model = model.cuda()

        optimizer = torchreid.optim.build_optimizer(model, optim='adam', lr=0.0003)
        scheduler = torchreid.optim.build_lr_scheduler(optimizer, lr_scheduler='single_step', stepsize=20)
        engine = torchreid.engine.VideoSoftmaxEngine(datamanager, model, optimizer=optimizer, scheduler=scheduler)

        print(f"\n=> Training {m_name} for 40 Epochs...")
        try:
            engine.run(save_dir=f'/kaggle/working/logs/{m_name}', max_epoch=40, eval_freq=0, print_freq=100)
        except AssertionError:
            print("=> Bypassed evaluation error successfully!")
            
        # =====================================================================
        # CRITICAL STEP: MANUALLY SAVE WEIGHTS FROM GPU TO DISK
        # =====================================================================
        save_path = f'/kaggle/working/logs/{m_name}/{m_name}_weights_final.pth'
        
        # Unwrap DataParallel layer (if any) to get the original model core
        state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
        torch.save(state_dict, save_path)
        print(f"=> [SUCCESS] MANUALLY SAVED MODEL TO DISK: {save_path}")
        # =====================================================================

        del model, optimizer, engine, datamanager
        torch.cuda.empty_cache()

    print("\n=> Packing zip file...")
    shutil.make_archive('/kaggle/working/resnet50mid_models', 'zip', '/kaggle/working/logs')
    print(">>> PROCESS COMPLETED SUCCESSFULLY! Models are safely archived.")